In [5]:
# metro_utils.py
import pandas as pd

def load_data():
    df = pd.read_csv("C:/Users/hp/metro/Delhi metro.csv")
    df["Station Names"] = df["Station Names"].str.strip().str.lower()
    return df

def get_line_map(df):
    return df.groupby("Metro Line")["Station Names"].apply(list).to_dict()


In [6]:
# logger functions defined here instead of importing
import time

def log_event(msg):
    with open("metro_log.txt", "a") as f:
        f.write(f"{time.ctime()} - {msg}\n")

def view_log():
    try:
        with open("metro_log.txt", "r") as f:
            print("\n--- System Log ---")
            print(f.read())
    except FileNotFoundError:
        print("No logs found.")


In [7]:
# train_simulation.py
import threading
import time


station_lock = threading.Lock()

def run_train(line_name, train_id, stations):
    for station in stations:
        with station_lock:
            log_event(f"Train {train_id} arrived at {station.title()} ({line_name})")
            print(f"[Train {train_id}] at {station.title()}")
        time.sleep(1)
    print(f"Train {train_id} finished {line_name}")

def simulate_trains():
    df = load_data()
    line_map = get_line_map(df)
    threads = []

    for i, line in enumerate(list(line_map.keys())[:2]):  # simulate 2 lines
        t = threading.Thread(target=run_train, args=(line, i+1, line_map[line]))
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    print("Train simulation complete.")


In [8]:
# scheduler.py
def run_fcfs():
    df = load_data()
    line_map = get_line_map(df)
    print("\nRunning FCFS Scheduler...")
    for i, (line, stations) in enumerate(line_map.items()):
        print(f"\n[Train {i+1}] on {line} Line")
        for station in stations[:5]:  # simulate part of line
            print(f"  -> {station.title()}")
            log_event(f"Train {i+1} (FCFS) passed {station.title()}")
            time.sleep(0.5)

def run_rr(time_slice=2):
    df = load_data()
    line_map = get_line_map(df)
    trains = list(line_map.items())[:3]
    max_len = max(len(stations) for _, stations in trains)

    print("\nRunning Round Robin Scheduler...")
    for i in range(max_len):
        for t_id, (line, stations) in enumerate(trains):
            if i < len(stations):
                station = stations[i]
                print(f"Train {t_id+1} on {line} -> {station.title()}")
                log_event(f"Train {t_id+1} (RR) passed {station.title()}")
                time.sleep(0.5)


In [9]:
# deadlock_simulation.py
lock1 = threading.Lock()
lock2 = threading.Lock()

def train_a():
    with lock1:
        print("Train A acquired Track A")
        time.sleep(1)
        with lock2:
            print("Train A acquired Track B")

def train_b():
    with lock2:
        print("Train B acquired Track B")
        time.sleep(1)
        with lock1:
            print("Train B acquired Track A")

def simulate_deadlock():
    print("\nSimulating Deadlock...")
    t1 = threading.Thread(target=train_a)
    t2 = threading.Thread(target=train_b)
    t1.start()
    t2.start()
    t1.join()
    t2.join()
    print("Deadlock simulation complete (may hang)")


In [10]:
# logger.py
import time

def log_event(msg):
    with open("metro_log.txt", "a") as f:
        f.write(f"{time.ctime()} - {msg}\n")

def view_log():
    try:
        with open("metro_log.txt", "r") as f:
            print("\n--- System Log ---")
            print(f.read())
    except FileNotFoundError:
        print("No logs found.")

In [11]:
def run_priority_scheduling():
    print("\nRunning Priority Scheduling...")

    trains = [
        {"id": 1, "line": "Yellow Line", "priority": 2},
        {"id": 2, "line": "Blue Line", "priority": 1},
        {"id": 3, "line": "Red Line", "priority": 3}
    ]

    # Sort trains by priority (lower value = higher priority)
    sorted_trains = sorted(trains, key=lambda t: t["priority"])

    df = load_data()
    line_map = get_line_map(df)

    for train in sorted_trains:
        stations = line_map.get(train["line"].lower())
        if not stations:
            print(f"Line {train['line']} not found!")
            continue
        print(f"\n[Train {train['id']}] {train['line']} (Priority {train['priority']})")
        for s in stations[:5]:
            print(f"  -> {s.title()}")
            log_event(f"Train {train['id']} (Priority {train['priority']}) at {s.title()}")
            time.sleep(0.5)


In [12]:
from threading import Semaphore

station_capacity = Semaphore(2)  # Only 2 trains can be at a station

def station_entry(train_id, station):
    with station_capacity:
        print(f"Train {train_id} entering {station.title()}")
        log_event(f"Train {train_id} entered {station.title()}")
        time.sleep(1)
        print(f"Train {train_id} leaving {station.title()}")
        log_event(f"Train {train_id} left {station.title()}")

def simulate_station_limit():
    print("\nSimulating Station Entry with Limited Capacity...")
    stations = ['rajiv chowk', 'karol bagh', 'kashmere gate']

    threads = []
    for i in range(5):
        t = threading.Thread(target=station_entry, args=(i+1, stations[i % 3]))
        threads.append(t)
        t.start()

    for t in threads:
        t.join()


In [13]:
import random

def simulate_interrupt():
    df = load_data()
    all_stations = df["Station Names"].tolist()
    blocked_station = random.choice(all_stations)
    print(f"\n⚠️ Maintenance Alert! {blocked_station.title()} is temporarily closed!")

    line_map = get_line_map(df)
    for i, (line, stations) in enumerate(line_map.items()):
        print(f"\n[Train {i+1}] {line}")
        for s in stations[:5]:
            if s == blocked_station:
                print(f"🚧 Skipping {s.title()} (Under Maintenance)")
                log_event(f"{s.title()} skipped due to interrupt")
            else:
                print(f"  -> {s.title()}")
                log_event(f"Train {i+1} passed {s.title()}")
                time.sleep(0.5)


In [ ]:
def main_menu():
    while True:
        print("\n==== Delhi Metro OS Project ====")
        print("1. Simulate Train Movement (Threads)")
        print("2. FCFS Scheduling")
        print("3. Round Robin Scheduling")
        print("4. Deadlock Simulation")
        print("5. View Log")
        print("6. Priority Scheduling")
        print("7. Station Capacity Simulation")
        print("8. Maintenance Interrupt Simulation")
        print("9. (Reserved for future feature)")  # Optional placeholder
        print("10. Exit")

        choice = input("Enter choice: ").strip()

        if choice == '1':
            simulate_trains()
        elif choice == '2':
            run_fcfs()
        elif choice == '3':
            run_rr()
        elif choice == '4':
            simulate_deadlock()
        elif choice == '5':
            view_log()
        elif choice == '6':
            run_priority_scheduling()
        elif choice == '7':
            simulate_station_limit()
        elif choice == '8':
            simulate_interrupt()
        elif choice == '10':
            print("Exiting. Thank you!")
            break
        else:
            print("❌ Invalid choice. Please enter a number between 1 and 10.")

# ✅ Run the menu
main_menu()



==== Delhi Metro OS Project ====
1. Simulate Train Movement (Threads)
2. FCFS Scheduling
3. Round Robin Scheduling
4. Deadlock Simulation
5. View Log
6. Priority Scheduling
7. Station Capacity Simulation
8. Maintenance Interrupt Simulation
9. (Reserved for future feature)
10. Exit


Enter choice:  88


❌ Invalid choice. Please enter a number between 1 and 10.

==== Delhi Metro OS Project ====
1. Simulate Train Movement (Threads)
2. FCFS Scheduling
3. Round Robin Scheduling
4. Deadlock Simulation
5. View Log
6. Priority Scheduling
7. Station Capacity Simulation
8. Maintenance Interrupt Simulation
9. (Reserved for future feature)
10. Exit
